# ExploreSat – Download & Explore Satellite Data

This notebook shows how to:
1. Download Sentinel-2 imagery from **Microsoft Planetary Computer** (free, no account)
2. Download Sentinel-2 / SRTM DEM from **Google Earth Engine** (free account)
3. Tile the imagery into training patches
4. Visualise a sample tile and its segmentation prediction

In [ ]:
import sys, pathlib
# Add the src directory to the path
sys.path.insert(0, str(pathlib.Path().resolve().parent / 'src'))

## 1. Microsoft Planetary Computer (no sign-in required)

In [ ]:
from exploresat.data.download import PlanetaryComputerDownloader

dl = PlanetaryComputerDownloader(dest_dir='../data/raw')

# Bounding box: Delhi, India  (lon_min, lat_min, lon_max, lat_max)
BBOX = (77.0, 28.4, 77.4, 28.8)

saved = dl.download_sentinel2(
    bbox=BBOX,
    date_range=('2023-06-01', '2023-09-30'),
    max_cloud_cover=20,
    max_items=2,
    bands=['B04', 'B03', 'B02'],   # RGB only
)
print('Downloaded:', saved)

## 2. Google Earth Engine (requires free account + `earthengine authenticate`)

In [ ]:
# Uncomment and fill in your GCP project ID
# from exploresat.data.download import GEEDownloader
# dl_gee = GEEDownloader(project='YOUR-GCP-PROJECT', dest_dir='../data/raw')
# dl_gee.download_sentinel2(bbox=BBOX, date_range=('2023-01-01','2023-12-31'))
# dl_gee.download_srtm_dem(bbox=BBOX)

## 3. Tile into 512×512 training patches

In [ ]:
from exploresat.data.download import tile_geotiff
import pathlib

tif_files = sorted(pathlib.Path('../data/raw/sentinel2').glob('*.tif'))
for tif in tif_files:
    tile_geotiff(
        src_path=tif,
        dest_dir=pathlib.Path('../data/processed') / tif.stem,
        tile_size=512,
        overlap=64,
    )

## 4. Visualise a downloaded scene

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt

def show_rgb(path, title=''):
    with rasterio.open(path) as src:
        data = src.read([1, 2, 3]).astype(float)
    # Percentile stretch
    for i in range(3):
        lo, hi = np.percentile(data[i], [2, 98])
        data[i] = np.clip((data[i] - lo) / max(hi - lo, 1e-6), 0, 1)
    rgb = np.transpose(data, (1, 2, 0))
    plt.figure(figsize=(8, 8))
    plt.imshow(rgb)
    plt.title(title or str(path))
    plt.axis('off')
    plt.show()

if saved:
    show_rgb(saved[0], 'Sentinel-2 RGB – Delhi')

## 5. Quick inference with the pre-trained model (after training)

In [ ]:
import pathlib, torch
CKPT = pathlib.Path('../checkpoints/best_model.pth')

if CKPT.exists() and saved:
    from exploresat.models.segmentation import build_model
    from exploresat.inference.predictor import Predictor
    from exploresat.data.dataset import class_to_rgb_mask, CLASS_NAMES

    model = build_model(encoder_weights=None)
    ckpt = torch.load(str(CKPT), map_location='cpu', weights_only=True)
    model.load_state_dict(ckpt['model_state_dict'])

    pred_path = pathlib.Path('../data/predictions/notebook_pred.tif')
    predictor = Predictor(model=model)
    predictor.predict_geotiff(saved[0], pred_path, export_rgb=True)

    show_rgb(pred_path.with_name(pred_path.stem + '_rgb.tif'), 'Segmentation result')
    print('Classes:', CLASS_NAMES)
else:
    print('Train the model first with: python scripts/train.py')

## 6. Start the visualisation server

Run the following in a terminal (not in this notebook):

```bash
python scripts/serve.py
```

Then open http://localhost:8000/ in your browser, or add the XYZ tile URL
to QGIS:
```
http://localhost:8000/tiles/{layer}/{z}/{x}/{y}.png
```